#Scrapping Data from news agencies usinng SerpAPI

#Mounnting google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Python is working!
Python version: 3.14.0 (tags/v3.14.0:ebf955d, Oct  7 2025, 10:15:03) [MSC v.1944 64 bit (AMD64)]


#Importing Libraries

In [ ]:
!pip install -q --no-cache-dir google-search-results
import importlib, sys, pkgutil, subprocess, json

#Establising SerpAPI connection

In [ ]:
# ✅ SerpAPI via plain HTTP (works on Py 3.12 too)
!pip -q install pandas requests

import os, requests, pandas as pd

API_KEY = os.getenv("SERPAPI_API_KEY") or "MpAPI" #Replace Second argument to your API
assert API_KEY != "YOUR_KEY_HERE", "Set SERPAPI_API_KEY or paste your key."

BASE = "https://serpapi.com/search.json"

def serpapi_search(engine: str, params: dict):
    r = requests.get(BASE, params={"engine": engine, "api_key": API_KEY, **params}, timeout=30)
    r.raise_for_status()
    return r.json()

print("✅ Setup complete")

#Through google news, particular news website 100 news articles are scrapped
#cnn.com
#bbc.com
#bytimes.com
#theguardian.com
#dw.com
#aljazeera.com
#ansa.it
#libero.it
#ft.com
#ilfattoquotidiano.it
#repubblica.it
#corriera.it

In [ ]:
# 📰 Google News — fetch up to 100 results from corriere.it
all_news = []
MAX_PAGES = 5  # 5 × 20 = 100

for p in range(MAX_PAGES):
    news = serpapi_search("google_news", {
        "q": "Venice Italy site:ft.com", # < --- WEbssite to scrap news from
        "hl": "en",
        "gl": "it",
        "start": p * 20  # pagination offset
    })

    items = news.get("news_results", [])
    if not items:
        break

    for r in items:
        all_news.append({
            "position": r.get("position"),
            "title": r.get("title"),
            "link": r.get("link"),
            "source": r.get("source"),
            "date": r.get("date"),
            "snippet": r.get("snippet"),
            "thumbnail": r.get("thumbnail"),
        })

df_news = pd.DataFrame(all_news)
print("Total news collected:", len(df_news))


#Print data frame

In [ ]:
df_news.head(100)

#Install and import libraries

In [ ]:
!pip install -q newspaper3k nltk
import nltk
nltk.download("vader_lexicon")
nltk.download("punkt")

#Extract Artioicle content

In [ ]:
from newspaper import Article
import time

def extract_full_text(url):
    try:
        article = Article(url)
        article.download()
        article.parse()
        return article.text
    except:
        return None

df_news["content"] = None

for i, link in enumerate(df_news["link"]):
    print(f"[{i+1}/{len(df_news)}] Extracting:", link)

    text = extract_full_text(link)
    df_news.at[i, "content"] = text

    time.sleep(1)   # avoid rate limiting

print("✅ Full article extraction complete.")

#Save data frame

In [ ]:
# 💾 Save your DataFrame: change df_name and path
df_name = 'df_news'   # replace with the variable name you want to save
OUTPATH = "/content/drive/MyDrive/Venice_RC15/Datasets/News/Venice_ftcom.csv"
globals()[df_name].to_csv(OUTPATH, index=False)
OUTPATH